# Ours (agreement + escalation) vs the T10/T11 judges: override off, inlier inflation applied to everything
Question: does our current best rule hold up against Palash's T10 rules when the same stress test is applied to ours, and with his 12-inlier override removed?
Data: our dumps official150 (in-sample for the frozen T7 judge) and nudt707 (150 photos; clean for the T7 judge, but 46 of them are in his T8 707 set and 46 in his 808 set).
Rules: **bars** (no agreement), **ours** (bar 8 or agreement, else 27 queries at bar 11), **J0** frozen T7 judge cut 0.9 + override 12, **J1** same without override, **J2** cut 0.98 without override (wide cut 0.98),
**ours OR J1** (our answer, else J1's), **V5-small** (relative inliers + scale augmentation, trained on the OTHER of our two sets: 150 photos, diagnostic only).
Not possible here: his V5/S1z as trained (need the NUDT design signal tables or the satellite-made photos, not in git). Inflation f: every inlier count x f (rounded), for every rule.

In [1]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import joblib, json, numpy as np, pandas as pd
pd.set_option("display.width", 220)
from train.checks import replay
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
T7 = ROOT / "tests/Test_Iteration_7/results"
SETS = {"official150": "official150", "nudt707": "nudt_fresh707"}
photos = {s: replay.load_set(T7 / "rich_dump" / f) for s, f in SETS.items()}
raw = pd.concat([pd.read_csv(T7 / f"signals_ours_{s}.csv.gz") for s in SETS], ignore_index=True)
judge = joblib.load(T7 / "judge_t7.joblib")
F = cj.FEATURES_CTX
SCALES = [1.0, 1.25, 1.5, 2.0]
print({s: len(p) for s, p in photos.items()}, len(raw))

{'official150': 150, 'nudt707': 150} 81000


In [2]:
def ours_rule(f):
    rows = []
    for data, ph in photos.items():
        for pid, photo in ph.items():
            inflated = [np.round(np.asarray(q["inliers"], float) * f) for q in photo["queries"]]
            r = replay.decide_current(photo, inliers=inflated)
            rows.append(dict(data=data, photo=pid, stage=r["stage"], pts=int(r["points"]), err_sigma=r["err_sigma"], n_queries=r["n_queries"]))
    return pd.DataFrame(rows)

def merge_or(a, b):
    j = a.set_index(["data", "photo"]).join(b.set_index(["data", "photo"]), rsuffix="_j")
    A, B = j.stage != "skip", j.stage_j != "skip"
    pts = np.where(A, j.pts, np.where(B, j.pts_j, 0))
    st = np.where(A, j.stage, np.where(B, j.stage_j, "skip"))
    return pd.DataFrame(dict(pts=pts, stage=st, n_queries=np.where(A, j.n_queries, np.maximum(j.n_queries, j.n_queries_j))), index=j.index).reset_index()

def judge_rules(t, p):
    out = {}
    out["bars (no agreement)"] = cj.baseline(t, 8, 11, min_choice=3)
    out["J0 judge+override12 (0.9)"] = cj.decide(t, p, 0.9, override=12, wide_tau=0.95, min_choice=3)
    out["J1 judge, no override (0.9)"] = cj.decide(t, p, 0.9, override=None, wide_tau=0.95, min_choice=3)
    out["J2 judge, no override (0.98)"] = cj.decide(t, p, 0.98, override=None, wide_tau=0.98, min_choice=3)
    return out

In [3]:
import gc
RESULT = {}
for f in SCALES:
    t = sr.scale_signals(raw, f)
    p = sr.predict(judge, t, F)
    r = judge_rules(t, p)
    r["ours (agreement+escalation)"] = ours_rule(f)
    r["ours OR J1"] = merge_or(r["ours (agreement+escalation)"], r["J1 judge, no override (0.9)"])
    RESULT[f] = r
    del t, p; gc.collect()
rows = []
for f, r in RESULT.items():
    for k, v in r.items():
        s = cj.summary(v)
        for data in s.index:
            rows.append(dict(scale=f, rule=k, data=data, score=s.loc[data, "score"], wrong=int(s.loc[data, "wrong"]), answered=s.loc[data, "answered"], queries=s.loc[data, "queries"]))
TAB = pd.DataFrame(rows)
for data in SETS:
    d = TAB[TAB.data == data]
    print(data)
    display(pd.concat({"score": d.pivot_table(index="rule", columns="scale", values="score").round(2), "wrong": d.pivot_table(index="rule", columns="scale", values="wrong").astype(int)}, axis=1))

official150


score                      wrong               
scale                         1.00   1.25   1.50   2.00  1.00 1.25 1.50 2.00
rule                                                                        
J0 judge+override12 (0.9)     56.8  56.80  57.20  57.20     0    0    0    0
J1 judge, no override (0.9)   56.8  56.80  57.20  57.20     0    0    0    0
J2 judge, no override (0.98)  54.0  54.24  55.04  55.84     0    0    0    0
bars (no agreement)           50.0  53.20  54.00  56.00     0    0    0    0
ours (agreement+escalation)   52.8  54.40  54.80  56.40     0    0    0    0
ours OR J1                    57.2  57.20  57.60  57.60     0    0    0    0

nudt707


score                      wrong               
scale                          1.00   1.25   1.50   2.00  1.00 1.25 1.50 2.00
rule                                                                         
J0 judge+override12 (0.9)     53.44  53.44  53.44  53.44     1    1    1    2
J1 judge, no override (0.9)   53.44  53.44  53.44  53.44     1    1    1    1
J2 judge, no override (0.98)  54.24  54.24  54.24  54.24     0    0    0    0
bars (no agreement)           53.04  53.84  51.44  49.84     1    1    4    6
ours (agreement+escalation)   53.84  53.84  51.44  49.84     1    1    4    6
ours OR J1                    54.24  54.24  51.44  49.84     1    1    4    6

In [4]:
# V5-small: relative inliers + scale augmentation, trained on the other set (150 photos), cut 0.9, no override. Diagnostic only.
FR = [("rli" if x == "li" else "rln22" if x == "ln22" else x) for x in F]
AUG = [0.5, 0.75, 1.0, 1.5, 2.0]
rows = []
for train_set, test_set in [("official150", "nudt707"), ("nudt707", "official150")]:
    lv = sr.far_levels(cj.add_context(cj.add_features(raw[raw.data == train_set])))
    prior = dict(far_inl=float(lv.far_inl.mean()), far_n22=float(lv.far_n22.mean()))
    base = cj.add_context(cj.add_features(raw))
    model, _, _ = sr.fit_judge(base, [train_set], FR, {"rli", "z", "hull", "cover9"}, AUG, lambda x: sr.add_relative(x, prior, 30))
    for f in SCALES:
        t = sr.scale_signals(raw[raw.data == test_set], f)
        tr = sr.add_relative(t, prior, 30)
        d = cj.decide(tr, sr.predict(model, tr, FR), 0.9, override=None, wide_tau=0.95, min_choice=3)
        s = cj.summary(d).loc[test_set]
        rows.append(dict(trained_on=train_set, test=test_set, scale=f, score=s.score, wrong=int(s.wrong), answered=s.answered, queries=s.queries))
V5 = pd.DataFrame(rows); V5

,trained_on,test,scale,score,wrong,answered,queries
0,official150,nudt707,1.00,51.68,3,0.927,8.10
1,official150,nudt707,1.25,51.68,3,0.927,8.10
2,official150,nudt707,1.50,51.68,3,0.927,8.10
3,official150,nudt707,2.00,51.68,3,0.927,8.10
4,nudt707,official150,1.00,54.00,0,0.913,8.38
5,nudt707,official150,1.25,54.00,0,0.913,8.38
6,nudt707,official150,1.50,54.00,0,0.913,8.38
7,nudt707,official150,2.00,54.24,0,0.920,8.38


In [5]:
# wrong answers per photo at x1 for every rule: which photos are the mistakes?
x1 = RESULT[1.0]
mis = []
for k, v in x1.items():
    w = v[(v.stage != "skip") & (v.pts < 0)] if "pts" in v else v[0:0]
    for _, r in w.iterrows():
        mis.append(dict(rule=k, data=r["data"], photo=r["photo"]))
pd.DataFrame(mis)

,rule,data,photo
0,bars (no agreement),nudt707,1181
1,J0 judge+override12 (0.9),nudt707,1181
2,"J1 judge, no override (0.9)",nudt707,1181
3,ours (agreement+escalation),nudt707,1181
4,ours OR J1,nudt707,1181


In [6]:
TAB.to_csv(T7 / "ours_vs_t10_scales.csv", index=False); V5.to_csv(T7 / "ours_vs_t10_v5small.csv", index=False)